In [1]:
import gc

import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow_f_i

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

#run_cv_tracked_mlflow_f_i(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train_kui + prev_application + installment",run_pfi=True)

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_global.csv")

#X_cleaned= clean_importance_zero_and_negative_pfi(importance_df,X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train_kui + prev_application + installment")

#cleaned = clean_importance_zero_and_negative_pfi(importance_df,merged_df)
#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

eliminando ['sellerplace_area_prev_3', 'obs_60_cnt_social_circle', 'instalments_days_of_underpayment_max_prev_1', 'log_diff_application_credit_min', 'applications_count', 'rate_down_payment_min', 'ratio_credit_to_goods_std', 'instalments_amt_instalment_median_prev_1', 'log_amt_goods_price_mean', 'organization_type_Transport: type 3', 'amt_application_median', 'building_score_std', 'instalments_amt_payment_max_prev_2', 'instalments_is_delinquency_sum_prev_3', 'name_yield_group_prev_1', 'week_appr_process_start_prev_2', 'ratio_credit_to_goods_prev_2', 'instalments_amt_payment_median_prev_2', 'instalments_repeated_for_underpayment_mean_prev_3', 'cnt_payment_prev_2', 'instalments_days_of_underpayment_max_prev_2', 'instalments_amt_instalment_sum_prev_2', 'instalments_log_amt_payment_mean_prev_2', 'instalments_amt_instalment_sum_max', 'total_interest_charged_prev_3', 'organization_type_Industry: type 5', 'name_portfolio_prev_1', 'name_portfolio_prev_3', 'organization_type_Police', 'name_type

115

In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)


#run_cv_tracked_mlflow_f_i(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"bureau_with_balance",run_pfi=True)

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_global.csv")

#X_cleaned= clean_importance_zero_and_negative_pfi(importance_df,X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau+bureau_balance")

#cleaned = clean_importance_zero_and_negative_pfi(importance_df,merged_df)
#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_bureau_with_balance.parquet")

#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72656
[1]	validation_0-auc:0.73578
[2]	validation_0-auc:0.73910
[3]	validation_0-auc:0.74179
[4]	validation_0-auc:0.74397
[5]	validation_0-auc:0.74643
[6]	validation_0-auc:0.74734
[7]	validation_0-auc:0.74874
[8]	validation_0-auc:0.75025
[9]	validation_0-auc:0.75208
[10]	validation_0-auc:0.75284
[11]	validation_0-auc:0.75531
[12]	validation_0-auc:0.75733
[13]	validation_0-auc:0.75812
[14]	validation_0-auc:0.75880
[15]	validation_0-auc:0.75977
[16]	validation_0-auc:0.76042
[17]	validation_0-auc:0.76083
[18]	validation_0-auc:0.76130
[19]	validation_0-auc:0.76179
[20]	validation_0-auc:0.76174
[21]	validation_0-auc:0.76199
[22]	validation_0-auc:0.76258
[23]	validation_0-auc:0.76299
[24]	validation_0-auc:0.76320
[25]	validation_0-auc:0.76342
[26]	validation_0-auc:0.76402
[27]	validation_0-auc:0.76414
[28]	validation_0-auc:0.76429
[29]	validation_0-auc:0.76501
[30]	validation_0-auc:0.76535
[31]	validation_0-auc:0.76513
[32]	validation_0-auc:0.76520
[33]	validation_0-au

3255

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")


merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

In [5]:
#app_train_with_feature_engineering + prev_app + bureau + installments
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "pruned_bureau_with_balance.parquet")

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments.parquet")

merged_df = previous_application_df.merge(
    bureau_df,
    on= "id_curr", 
    how="left"
)

merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith('_y')].rename(columns=lambda x: x.rstrip('_x'))

#cleaning the third
del previous_application_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72929
[1]	validation_0-auc:0.73938
[2]	validation_0-auc:0.74514
[3]	validation_0-auc:0.74717
[4]	validation_0-auc:0.74957
[5]	validation_0-auc:0.75220
[6]	validation_0-auc:0.75520
[7]	validation_0-auc:0.75739
[8]	validation_0-auc:0.75943
[9]	validation_0-auc:0.76107
[10]	validation_0-auc:0.76306
[11]	validation_0-auc:0.76521
[12]	validation_0-auc:0.76701
[13]	validation_0-auc:0.76785
[14]	validation_0-auc:0.76926
[15]	validation_0-auc:0.77011
[16]	validation_0-auc:0.77162
[17]	validation_0-auc:0.77202
[18]	validation_0-auc:0.77208
[19]	validation_0-auc:0.77254
[20]	validation_0-auc:0.77278
[21]	validation_0-auc:0.77354
[22]	validation_0-auc:0.77380
[23]	validation_0-auc:0.77397
[24]	validation_0-auc:0.77412
[25]	validation_0-auc:0.77426
[26]	validation_0-auc:0.77456
[27]	validation_0-auc:0.77436
[28]	validation_0-auc:0.77492
[29]	validation_0-auc:0.77577
[30]	validation_0-auc:0.77635
[31]	validation_0-auc:0.77632
[32]	validation_0-auc:0.77570
[33]	validation_0-au

409